# OpenAI Responses API Client
Generic client for testing any OpenAI-compatible `/v1/responses` endpoint.
The server handles the agentic tool loop — client sends one request, gets final answer.

In [ ]:
import httpx
import json

BASE_URL = "http://localhost:9000"
ENDPOINT = f"{BASE_URL}/v1/responses"

## 1. Single request (no tools)

In [ ]:
resp = httpx.post(ENDPOINT, json={
    "model": "GPT OSS 120B",
    "input": [{"role": "user", "content": "Say hello in one word."}]
}, timeout=60)

print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

## 2. Real MCP server (Parallel Search) — server runs the loop, returns final answer

This exercises the full agentic loop against a **live, public, no-auth MCP server**:
[Parallel Search MCP](https://docs.parallel.ai/integrations/mcp/search-mcp)
(`https://search.parallel.ai/mcp`, streamable-HTTP).

The MCP tools are configured **server-side**, not sent by the client. `mcp.json`
in the project root is auto-loaded (it's the default `mcp_config_path`), so just
start the server:

```bash
CHAT2API_PROVIDER=expressai CHAT2API_HEADLESS=false python -m src.main
```

On startup the proxy connects, lists Parallel's tools, and advertises them as
`parallel__web_search` / `parallel__web_fetch`. For a query needing search, the
model emits tool calls, the **proxy executes them against Parallel** (parallel
calls run concurrently), feeds the results back, and loops until a final answer —
the client just sees the final answer plus the `mcp_call` trace.

In [ ]:
# No client-side `tools` here — the Parallel Search MCP tools come from the
# server config (mcp.json). The proxy runs the whole loop and returns the final
# answer plus the mcp_call trace.
# `model` must be one the active provider offers (see GET /v1/models).
resp = httpx.post(ENDPOINT, json={
    "model": "GPT OSS 120B",
    "input": [{"role": "user", "content":
        "Use web search to find who won the most recent Formula 1 race and on what date. "
        "Cite the source URL."}],
}, timeout=180)

data = resp.json()
print(f"Status: {data.get('status')}\n")

# Show the tool trace the proxy executed on our behalf, then the final answer.
for item in data.get("output", []):
    if item.get("type") == "mcp_call":
        out = str(item.get("output", ""))[:200].replace("\n", " ")
        print(f"[mcp_call] {item['name']}({item['arguments']}) => {out}…")

print("\nFinal answer:\n", data.get("output_text"))

## 3. Error handling

In [ ]:
resp = httpx.post(ENDPOINT, json={"invalid": True}, timeout=10)
print(resp.status_code)
print(resp.json())

## 4. Health check

In [ ]:
try:
    resp = httpx.get(f"{BASE_URL}/docs", timeout=5)
    print(f"Server up: {resp.status_code == 200}")
except Exception as e:
    print(f"Server not reachable: {e}")